In [ ]:
import pandas as pd

import config as c

In [ ]:
# build PR-BD corpus

# 1) posts in BD subreddits
# subreddit_type based on categorisation here: https://github.com/glorisonne/reddit_bd_mood_posting_mh/blob/main/data/subreddit_topics.csv
# (Fourth level = "bipolar")
#posts = pd.read_csv(c.data + "posts_meta.csv", usecols=["id", "user_id", "subreddit_name", "text_wordcount", "mentions_bd",
#                                                        "subreddit_type"])

#posts = posts[posts.subreddit_type == "bd"]
#print("Posts in BD subreddits:\nPosts: %d\nWords: %d\nUsers: %d" %(len(posts), posts.text_wordcount.sum(),
#                                                                    posts.user_id.nunique()))

# 2) posts that mention BD -- refer to BD terms list https://github.com/glorisonne/reddit_bd_user_characteristics/blob/master/disclosure-patterns/condition-terms/bipolar-filter-terms.txt
#posts = posts[posts.mentions_bd]
#print("Posts that mention BD:\nPosts: %d\nWords: %d\nUsers: %d" %(len(posts), posts.text_wordcount.sum(),
#                                                                    posts.user_id.nunique()))

# 3) posts with at least 94 words, duplicates (same text by same user) removed
#posts = posts[posts.text_wordcount > 93]

# add post texts - read post_text csv in two batches as the file is very large and the code migth therefore fail on
# machines with less RAM
#posts_text_0 = pd.read_csv(c.data + "posts_text.csv", nrows=10000000)
#posts_text_0 = posts_text_0[posts_text_0.id.isin(posts.id)]
#posts_text_1 = pd.read_csv(c.data + "posts_text.csv", skiprows=10000000, header=None, names=["id", "text"])
#posts_text_1 = posts_text_1[posts_text_1.id.isin(posts.id)]
#posts_text = pd.concat([posts_text_0, posts_text_1])

#posts = posts.merge(posts_text, left_on="id", right_on="id")

#del(posts_text_0, posts_text_1, posts_text)

#posts.drop_duplicates(subset=["text", "user_id"], keep = "last", inplace=True)
#print("Posts with at least 94 words, duplicates removed:\nPosts: %d\nWords: %d\nUsers: %d" %(len(posts), posts.text_wordcount.sum(),
#                                                                    posts.user_id.nunique()))
posts.to_csv(c.data_local + "posts_bd.csv")

# TBD lemmatize (check when did this for the wordcount in the flowchart)
# preprocess_posts_with_spacy.py. nohup python process_posts_with_spacy.py data/posts_bd.csv &> output/process_posts_spacy.log &
comBine output files

# 4) cosine similarity with PR terms list - share PR terms list in data/ (ToDo: possibly share "standard" list with variants + list of only unique terms)
PR_scoring.py --> outputs posts_bd_spacy_phrases.csv und posts_bd_PR_scored.csv

# 5) Reference corpus - cosine similarity cutoff

In [ ]:
# figure out where different number of posts comes from for BD subreddit posts

# lower/uppercasing does not matter for the BD subreddits in subreddit_topics.csv
subreddits = pd.read_csv(c.data_local + "subreddit_topics.csv")
bd_subreddits = subreddits[subreddits["fourth level"] == "bipolar"]
posts_ignore_case = posts[posts.subreddit_name.str.lower().isin(bd_subreddits.subreddit.str.lower())]
posts_with_case = posts[posts.subreddit_name.isin(bd_subreddits.subreddit)]
print(len(posts_ignore_case), len(posts_with_case))

# but it does matter for the subreddits from bipolar-subreddits.txt
bd_subreddits_old = pd.read_csv(c.data_local + "bipolar-subreddits.txt", header=None, names=["subreddit"]).squeeze(axis=0)
print(len(bd_subreddits_old))
print(bd_subreddits)
posts_bd_old = posts[posts.subreddit_name.isin(bd_subreddits_old.subreddit)]
posts_bd_old_ignore_case = posts[posts.subreddit_name.str.lower().isin(bd_subreddits_old.subreddit.str.lower())]
print(len(posts_bd_old), len(posts_bd_old_ignore_case))

# reason: three subreddits have different upper/lowercasing in bipolar-subreddits.txt
print(bd_subreddits[~bd_subreddits.subreddit.isin(bd_subreddits_old.subreddit)].subreddit)
print(bd_subreddits_old[~bd_subreddits_old.subreddit.isin(bd_subreddits.subreddit)].subreddit)

In [ ]:
# check unique PR terms
PR_terms = pd.read_csv(c.data_local + "PR_terms_unique.csv")
#print(PR_terms)
PR_terms.variants.fillna("", inplace=True)
PR_terms["num_variants"] = PR_terms.variants.apply(lambda x: len(x.split(", ")))
print(PR_terms.num_variants.sum())

# separate row for each variant
PR_terms["variants"] = PR_terms.variants.apply(lambda x: x.split(", "))
PR_terms_expanded = PR_terms.explode('variants').reset_index(drop=True)
import numpy as np
PR_terms_expanded["variants"] = np.where(PR_terms_expanded.variants.str.len() == 0, PR_terms_expanded.term,
                                          PR_terms_expanded.variants)
#pd.set_option('display.max_rows', 500)
#print(PR_terms_expanded[["term", "variants"]].head(n=500))

# PR_terms_expanded_orig = pd.read_csv(c.data_local + "PR_terms.csv")
# print(PR_terms_expanded_orig)
# PR_terms_expanded_orig["term"] = PR_terms_expanded_orig.term.str.replace(" - ", "-")
#PR_terms_expanded_orig["term"] = PR_terms_expanded_orig.term.str.replace(" #", "#")
#PR_terms_expanded_orig["term"] = PR_terms_expanded_orig.term.str.replace("# ", "#")
# validate # yourself # is missing in orig
print(PR_terms_expanded[~PR_terms_expanded.variants.isin(PR_terms_expanded_orig.term)].variants.unique())
print(PR_terms_expanded_orig[~PR_terms_expanded_orig.term.isin(PR_terms_expanded.variants)].term.unique())

In [ ]:
# build corpora from scored posts
#posts = pd.read_csv(c.data_local + "posts_bd_PR_scored.csv", usecols=["post_id", "text", "text_with_phrases", "PR"])
#posts_meta = pd.read_csv(c.data_local + "posts_bd.csv", usecols=["id", "user_id", "subreddit_name", "text_wordcount"])
#print(posts_meta)
#posts = posts.merge(posts_meta, left_on="post_id", right_on="id", how="left")
# posts.drop(labels=["id"], axis=1,inplace=True)
# print(posts)

PR = posts[posts.PR > 0.025]
not_PR = posts[posts.PR < 0.013]
print("PR-BD Corpus:\nPosts: %d\nWords: %d\nUsers: %d" %(len(PR), PR.text_wordcount.sum(), PR.user_id.nunique()))
print("Comparison Corpus:\nPosts: %d\nWords: %d\nUsers: %d" %(len(not_PR), not_PR.text_wordcount.sum(), not_PR.user_id.nunique()))

# get different corpus sizes than when I did it for the first time because the dataset that I used as a basis for calculating
# the tf-idf scores was a different one
# also need to check if I can replicate my results if I first select only posts with at least 94 words and remove duplicates
# and then calculate the PR scores

write_corpus = True
# important! Need to remove corpus folder with individual files for user if want to re-create,
# otherwise files of users in previous corpus version remain!
if write_corpus:
    PR.to_csv(c.data_local + "PR-BD_Corpus.csv")
    with open(c.data_local + "PR-BD_Corpus.txt", "w") as f:
        f.write("\n".join(PR.text.tolist()))

    # write one file per user to calculate dispersion
    users = PR.groupby("user_id")
    for user_id, posts in users:
        with open(c.data_local + "/PR-BD_corpus/%s.txt" %user_id, "w") as f:
            f.write("\n".join(posts.text.tolist()))

    not_PR.to_csv(c.data_local + "Reference_Corpus.csv")
    with open(c.data_local + "Reference_Corpus.txt", "w") as f:
        f.write("\n".join(not_PR.text.tolist()))

In [ ]:
# calculate keyness + select key lemmas
# preprocess LancsBox keyword lists
# nead to read LancsBox output in as text files as they are not properly csv formatted (" not escaped)
def read_lacsbox_file(fname):
    lines = []
    with open(c.data_local + "%s" %fname) as f:
        # skip firs two lines
        f.readline()
        f.readline()
        for line in f:
            lines.append(line.strip().split("\t"))
    return lines
#lines_LL = read_lacsbox_file("PR-BD_key_lemmas_LL.txt")
#keywords_LL = pd.DataFrame(lines_LL, columns=["term", "frequency", "dispersion", "frequency_ref", "dispersion_ref", "LL"])
#keywords_LR = pd.DataFrame(read_lacsbox_file("PR-BD_key_lemmas_LR.txt"),
#                           columns=["term", "frequency_c", "dispersion_c", "frequency_ref", "dispersion_ref", "LR"])

print(len(keywords_LL))
print(len(keywords_LR))

# merge LR and LL scores
keywords = keywords_LL[["term", "frequency", "dispersion", "frequency_ref", "LL"]].\
                        merge(keywords_LR[["term", "LR"]], left_on="term", right_on="term", how="left")

# preprocess lemmas: i|be_pron|v -> split at first "_", take only first part to remove POS, then replace | by " "
keywords["lemma_pos"] = keywords.term

def extract_lemma(term):
    return term.split("_")[0].replace("|", " ")

keywords["term"] = keywords.term.apply(extract_lemma)

print(keywords)

#for col in ["frequency", "dispersion", "frequency_ref", "LL", "LR"]:
#    keywords[col] = keywords[col].astype(float)

#key_lemmas_selected = keywords[(keywords.LL > 15.13) & (keywords.LR >= 1.0) & (keywords.dispersion >= 5.0)]
print(key_lemmas_selected)

keywords.to_csv(c.data_local + "PR-BD_key_lemmas_all.csv")

key_lemmas_selected.to_csv(c.data_local + "PR-BD_key_lemmas.csv")

In [ ]:
# compare new + old key lemmas
#key_lemmas_old = pd.read_excel(c.data_local + "PR-BD_key_lemmas_new_old_comparison.xlsx", sheet_name = "old")
#key_lemmas_new = pd.read_excel(c.data_local + "PR-BD_key_lemmas_new_old_comparison.xlsx", sheet_name = "new")

print(key_lemmas_old[~key_lemmas_old.lemma_pos.isin(key_lemmas_new.lemma_pos)]\
      [["frequency", "dispersion", "frequency_ref", "LL", "LR", "lemma_pos"]])
print(key_lemmas_new[~key_lemmas_new.lemma_pos.isin(key_lemmas_old.lemma_pos)]\
     [["frequency", "dispersion", "frequency_ref", "LL", "LR", "lemma_pos"]])

In [ ]:
# select post ids + anonymise

folder = c.data_local + "../post_ids/original/"
folder_anonymised = folder + "../"

# anonymisation
#post_id_mapping = pd.read_csv(c.data + "post_id_mapping.csv")
#post_id_map = dict(zip(post_id_mapping.original_id, post_id_mapping.id))
def anonymise_post_ids(fname):
    posts = pd.read_csv(folder + fname)
    posts["id"] = posts.id.map(post_id_map)
    # remove "-original" from filename
    fname_anonymised = fname.split("-")[0] + ".csv"
    posts.to_csv(folder_anonymised + fname_anonymised, index=False)
    print("Anonymised %s" %fname)

# PR, clinical recovery (posts after *recover* coding)
anonymise_post_ids("PR_post_ids-original.csv")
anonymise_post_ids("clinical_recovery_post_ids-original.csv")
# PR-BD Corpus
# Reference Corpus
# BD subreddit corpus
# (SMHD comparison corpus)
# *recover* corpus
# Reference corpus for *recover* corpus (?)

# Posts coded for PR relevance + coding results
# posts_PR_relevance = pd.read_csv(folder + "posts_PR_relevant_post_ids_to_anonymise.csv")
#posts_PR_relevance.relevant.value_counts()
# anonymise_post_ids("posts_PR_relevant_post_ids-original.csv")